# Proposed pipeline — Finding-Conditioned CT Temporal Progression (Colab)

Implements the design in `docs/proposed_pipeline.png`:

**d_f = g(v_prior, v_current, finding)**

**What is new vs `train_ctclip_temporal_colab.ipynb` (current pipeline):**
1. **Finding conditioning** inside the Difference Transformer (`e_diff <- e_diff + e_f`, or finding as a 4th token) so one pair produces many finding-specific `d_f`.
2. **Masked same-finding SupCon** on `d_f` (not instance InfoNCE vs report text): positives = same finding + same direction; negatives = same finding + other direction; **other findings are ignored** (masked out).
3. **Separate contrastive temperature** `tau_con` (does not share CE `logit_scale`).
4. **Contrastive-aware batching** (~K findings x 3 classes) so each batch has usable SupCon positives/negatives.
5. **Ablation knobs:** CE | +mag | +SupCon · templates vs real text for prototypes · +/- finding conditioning.

**Unchanged protocol:**
- Frozen CT-CLIP image/text towers; only the Difference Transformer trains.
- Hub `train_*` -> train + patient-held-out tune (early-stop on macro-F1); Hub `valid_*` -> **one** final test.
- Inference: `argmax cos(d_f, PROTO[finding])` — **no report text** at test time.
- Prefer `report_explicit` / tier=`explicit` silver labels.

Diagram: `docs/proposed_pipeline.png`.

## 1. Setup: clone CT-CLIP + our repo, install deps

In [ ]:
%cd /content
![ -d CT-CLIP ] || git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
%cd /content/CT-CLIP
!pip install -q -e transformer_maskgit
!pip install -q -e CT_CLIP
!pip install -q nibabel scipy huggingface_hub transformers scikit-learn tqdm
%cd /content
import sys
for p in ['/content/CT-CLIP/CT_CLIP', '/content/CT-CLIP/transformer_maskgit']:
    if p not in sys.path: sys.path.insert(0, p)
![ -d 3dCT ] || git clone https://github.com/nprakash1/3dCT.git 3dCT
!cd 3dCT && git pull -q
sys.path.append('/content/3dCT/scripts')
import ct_clip, transformer_maskgit; print('CT-CLIP import OK')

## 2. Mount Drive + config paths + ablation knobs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, csv, math, random, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, confusion_matrix
csv.field_size_limit(10**9)

DRIVE      = '/content/drive/MyDrive/3dCT'
IMG_DIR    = f'{DRIVE}/ctclip_cache/img'             # 512-d per volume
WEIGHTS    = f'{DRIVE}/ctclip_weights'               # CT-CLIP_v2.pt
PROTO_PT   = f'{DRIVE}/ctclip_cache/proto_bank.pt'   # finding x class prototypes
LAB        = '/content/3dCT/medgemma_labels_v3.jsonl'
LAB_DS     = '/content/3dCT/medgemma_labels (2).jsonl'  # optional real temporal sentences

# Hub train = only development domain. Hub validation = final test only.
TUNE_FRAC  = 0.15
SPLIT_SEED = 2026
REQUIRE_COMPLETE_HUB_VALID_FEATURES = True

CLASSES = ['worsened', 'stable', 'improved']
C2I = {c: i for i, c in enumerate(CLASSES)}
I2C = {i: c for c, i in C2I.items()}

# -------------------- proposed-design ablation knobs --------------------
FINDING_CONDITIONING = True     # finding enters g(.); False = current-pipeline baseline
FINDING_AS_4TH_TOKEN = False    # False: e_diff += e_f; True: tokens [e_diff, tp, tc, e_f]
USE_LEARNED_FINDING_EMB = True  # True: nn.Embedding; False: freeze e_f from text tower name emb

USE_CE         = True           # always keep True for classification readout
USE_MAGNITUDE  = True           # optional BCE(mag, change vs stable)
USE_SUPCON     = True           # masked same-finding SupCon (NEW)

PROTO_SOURCE   = 'templates'    # 'templates' | 'real'
ANTISYM        = False          # d = g(c,p)-g(p,c); usually leave False with finding conditioning

D_MODEL, EPOCHS, LR, PATIENCE = 256, 120, 1e-3, 20
WEIGHT_DECAY = 1e-2
LAMBDA_CE, LAMBDA_MAG, LAMBDA_CON = 1.0, 0.5, 0.5
TAU_CON_INIT = 0.07             # separate SupCon temperature (not logit_scale)

# contrastive-aware batching: ~K findings x up to 3 classes x N_PER_CLASS rows
K_FINDINGS_PER_BATCH = 8
N_PER_CLASS          = 4        # target rows per (finding, class) bucket in a batch
MAX_BATCH_SIZE       = 256      # hard cap

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| img cache exists:', os.path.isdir(IMG_DIR))
print('knobs:', dict(
    FINDING_CONDITIONING=FINDING_CONDITIONING, FINDING_AS_4TH_TOKEN=FINDING_AS_4TH_TOKEN,
    USE_MAGNITUDE=USE_MAGNITUDE, USE_SUPCON=USE_SUPCON, PROTO_SOURCE=PROTO_SOURCE,
    ANTISYM=ANTISYM))

## 3. Load cached CT-CLIP IMAGE embeddings (512-d per volume)

In [ ]:
import glob
POOLED = {}
for fp in glob.glob(f'{IMG_DIR}/*.pt'):
    key = os.path.basename(fp)[:-3]
    POOLED[key] = torch.load(fp, map_location='cpu').float()  # (512,)
print('loaded image embeddings for', len(POOLED), 'volumes')
cache_domain = Counter('train' if k.startswith('train_') else
                       'hub_valid' if k.startswith('valid_') else 'unknown' for k in POOLED)
print('cache by CT-RATE Hub domain:', dict(cache_domain))
assert POOLED, 'No image embeddings found — run ctclip_features_colab.ipynb first.'
print('example dim:', tuple(next(iter(POOLED.values())).shape))

## 4. Frozen CT-CLIP text tower (prototypes + optional finding-name embeddings)

Weights stay frozen. Used only to build `PROTO[finding]` and (if `USE_LEARNED_FINDING_EMB=False`) a frozen finding-name vector for conditioning.

In [ ]:
from huggingface_hub import login, hf_hub_download
from ctclip_utils import CTCLIPEmbedder, REPO_ID, CTCLIP_WEIGHTS_HF
login()  # paste READ token if weights not already on Drive
os.makedirs(WEIGHTS, exist_ok=True)
wp = f'{WEIGHTS}/{os.path.basename(CTCLIP_WEIGHTS_HF)}'
if not os.path.exists(wp):
    wp = hf_hub_download(REPO_ID, CTCLIP_WEIGHTS_HF, repo_type='dataset', local_dir=WEIGHTS)
emb = CTCLIPEmbedder(wp)
print('CT-CLIP text tower ready on', emb.device)

## 5. Labels + strict Hub train / Hub validation separation

- Rows are `(pair, finding, y)` with `y in {worsened, stable, improved}`.
- Prefer tier=`explicit` (report_explicit).
- `train_*` -> train / tune (patient SHA partition). `valid_*` -> final test only.

In [ ]:
def vkey(v): return v.replace('.nii.gz', '').replace('.nii', '')

def volume_domain(v):
    k = vkey(v).lower()
    if k.startswith('train_'): return 'hub_train'
    if k.startswith('valid_'): return 'hub_valid'
    return 'unknown'

def pair_domain(pv, cv):
    a, b = volume_domain(pv), volume_domain(cv)
    return a if a == b else 'cross_domain'

def dev_partition(patient):
    raw = f'{SPLIT_SEED}|{patient}'.encode('utf-8')
    u = int.from_bytes(hashlib.sha256(raw).digest()[:8], 'big') / 2**64
    return 'tune' if u < TUNE_FRAC else 'train'

recs = {}
for line in open(LAB, encoding='utf-8'):
    if not line.strip(): continue
    x = json.loads(line)
    recs[(x['patient'], x['prior_volume'], x['curr_volume'])] = x

# optional real temporal / dynamic sentences (per pair)
dyn_of = {}
try:
    for line in open(LAB_DS, encoding='utf-8'):
        if not line.strip(): continue
        x = json.loads(line)
        ds = x.get('dynamic_sentences') or []
        if isinstance(ds, list): ds = ' '.join(s for s in ds if isinstance(s, str))
        dyn_of[(x['patient'], x['prior_volume'], x['curr_volume'])] = (ds or '').strip()
    print('loaded dynamic_sentences for', len(dyn_of), 'pairs')
except FileNotFoundError:
    print('WARN: dynamic-sentence file not found:', LAB_DS)

pair_ids = {}
examples = {'train': [], 'tune': [], 'test': []}
skipped = Counter()
candidate_pairs = Counter(); usable_pairs = Counter(); missing_hub_valid = []
for key, rec in recs.items():
    patient, pv, cv = key
    domain = pair_domain(pv, cv)
    if domain == 'hub_train':
        sp = dev_partition(patient)
    elif domain == 'hub_valid':
        sp = 'test'
    else:
        skipped[domain] += 1; continue
    if not rec.get('parse_ok'):
        skipped[f'no_label_{sp}'] += 1; continue
    candidate_pairs[sp] += 1
    if vkey(pv) not in POOLED or vkey(cv) not in POOLED:
        skipped[f'no_feature_{sp}'] += 1
        if sp == 'test': missing_hub_valid.append(key)
        continue
    usable_pairs[sp] += 1
    for fd in rec.get('findings', []):
        if fd.get('tier') != 'explicit':
            skipped['not_explicit'] += 1; continue
        d = fd.get('direction'); f = fd.get('finding')
        if d not in C2I or not f:
            skipped['bad_dir'] += 1; continue
        pid = pair_ids.setdefault(key, len(pair_ids))
        examples[sp].append({
            'vp': vkey(pv), 'vc': vkey(cv), 'patient': patient,
            'finding': f, 'hub_domain': domain,
            'y': C2I[d], 'pid': pid,
            'evidence': fd.get('evidence', '') or '',
            'dynamic': dyn_of.get(key, ''),
        })

for sp in ['train', 'tune']:
    assert all(e['hub_domain'] == 'hub_train' and e['vp'].startswith('train_') and
               e['vc'].startswith('train_') for e in examples[sp])
assert all(e['hub_domain'] == 'hub_valid' and e['vp'].startswith('valid_') and
           e['vc'].startswith('valid_') for e in examples['test'])
patients = {sp: {e['patient'] for e in examples[sp]} for sp in examples}
assert patients['train'].isdisjoint(patients['tune'])
assert patients['train'].isdisjoint(patients['test'])
assert patients['tune'].isdisjoint(patients['test'])
assert examples['train'] and examples['tune'] and examples['test']
if REQUIRE_COMPLETE_HUB_VALID_FEATURES:
    assert not missing_hub_valid, (
        f'{len(missing_hub_valid)} labeled Hub-validation pairs lack cached features; '
        f'finish encode_split("valid") first. First missing: {missing_hub_valid[:3]}')

print('candidate labeled pairs:', dict(candidate_pairs))
print('usable cached pairs    :', dict(usable_pairs))
for sp in ['train', 'tune', 'test']:
    cc = Counter(e['y'] for e in examples[sp])
    print(f'{sp:5}: {len(examples[sp]):5} ex / {len(patients[sp]):4} patients  '
          f'(worsened={cc[0]} stable={cc[1]} improved={cc[2]})')
print('skipped:', dict(skipped))
print('LEAKAGE CHECK PASSED: optimizer=train_* only; tune=train_* only; final test=valid_* only')

FINDINGS = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']
F2I = {f: i for i, f in enumerate(FINDINGS)}
assert {e['finding'] for sp in examples for e in examples[sp]} <= set(FINDINGS)
print('canonical findings:', len(FINDINGS))
for sp in examples:
    for e in examples[sp]:
        e['fid'] = F2I[e['finding']]

## 6. Build finding x class text prototypes (`PROTO[finding]`, 3x512, frozen)

Default = template bank (v1). Set `PROTO_SOURCE='real'` to average available evidence/dynamic text per (finding, class) from the **train** split only (falls back to templates if empty).

In [ ]:
TEMPLATES = {
    'worsened': ['{f} has increased compared to the prior study',
                 '{f} has worsened since the previous exam',
                 'interval enlargement of {f}', 'new {f}', 'increased {f}'],
    'stable':   ['{f} is unchanged compared to the prior study',
                 'stable {f} with no interval change',
                 'no significant change in {f}', '{f} appears similar to prior'],
    'improved': ['{f} has decreased compared to the prior study',
                 '{f} has improved since the previous exam',
                 'interval decrease of {f}', '{f} has resolved', 'decreased {f}'],
}

def l2np(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-8)

def embed_mean(texts):
    texts = [t for t in texts if t and str(t).strip()]
    if not texts:
        return None
    vecs = []
    B = 64
    for i in range(0, len(texts), B):
        vecs.append(emb.embed_texts(texts[i:i+B], normalize=True).numpy())
    e = np.concatenate(vecs, 0).mean(0)
    return l2np(e)

proto_cache = PROTO_PT if PROTO_SOURCE == 'templates' else PROTO_PT.replace('.pt', f'_{PROTO_SOURCE}.pt')

if os.path.exists(proto_cache) and PROTO_SOURCE == 'templates':
    PROTO = torch.load(proto_cache)
    print('loaded cached prototypes for', len(PROTO), 'findings from', proto_cache)
else:
    PROTO = {}
    from tqdm.auto import tqdm
    real_bank = defaultdict(lambda: defaultdict(list))
    if PROTO_SOURCE == 'real':
        for e in examples['train']:
            txt = (e.get('evidence') or e.get('dynamic') or '').strip()
            if txt:
                real_bank[e['finding']][e['y']].append(txt)
    for f in tqdm(FINDINGS):
        rows = []
        for ci, c in enumerate(CLASSES):
            if PROTO_SOURCE == 'real':
                vec = embed_mean(real_bank[f][ci])
                if vec is None:
                    prompts = [t.format(f=f.lower()) for t in TEMPLATES[c]]
                    vec = embed_mean(prompts)
            else:
                prompts = [t.format(f=f.lower()) for t in TEMPLATES[c]]
                vec = embed_mean(prompts)
            rows.append(vec)
        PROTO[f] = torch.tensor(np.stack(rows)).float()  # (3,512)
    torch.save(PROTO, proto_cache)
    print('built + cached prototypes for', len(PROTO), 'findings ->', proto_cache, '| source=', PROTO_SOURCE)

# optional frozen finding-name embeddings for conditioning (when not using learned emb)
FINDING_NAME_PT = f'{DRIVE}/ctclip_cache/finding_name_emb.pt'
if os.path.exists(FINDING_NAME_PT):
    FINDING_NAME_EMB = torch.load(FINDING_NAME_PT)
    print('loaded finding-name embeddings', FINDING_NAME_EMB.shape)
else:
    name_prompts = [f'finding: {f.lower()}' for f in FINDINGS]
    FINDING_NAME_EMB = emb.embed_texts(name_prompts, normalize=True).float().cpu()  # (18,512)
    torch.save(FINDING_NAME_EMB, FINDING_NAME_PT)
    print('built finding-name embeddings', FINDING_NAME_EMB.shape, '->', FINDING_NAME_PT)

## 7. Tensorize splits

In [ ]:
def tensorize(exs):
    VP = torch.stack([POOLED[e['vp']] for e in exs])
    VC = torch.stack([POOLED[e['vc']] for e in exs])
    PR = torch.stack([PROTO[e['finding']] for e in exs])          # (N,3,512)
    Y  = torch.tensor([e['y'] for e in exs], dtype=torch.long)
    FID = torch.tensor([e['fid'] for e in exs], dtype=torch.long)
    PID = torch.tensor([e['pid'] for e in exs], dtype=torch.long)
    Fn = [e['finding'] for e in exs]
    return {'vp': VP, 'vc': VC, 'pr': PR, 'y': Y, 'fid': FID, 'pid': PID, 'fn': Fn}

DATA = {sp: tensorize(examples[sp]) for sp in ['train', 'tune', 'test']}
cnt = Counter(DATA['train']['y'].tolist()); tot = sum(cnt.values())
W_cls = torch.tensor([tot / (3 * max(cnt[i], 1)) for i in range(3)], dtype=torch.float32)
print('class weights (w/s/i):', [round(x, 3) for x in W_cls.tolist()])
for sp in DATA:
    print(sp, {k: (tuple(v.shape) if torch.is_tensor(v) else len(v)) for k, v in DATA[sp].items()})

## 8. Trainable module — finding-conditioned Difference Transformer

Only this module is trained. CT-CLIP stays frozen.

- Inputs: `v_prior`, `v_current`, `finding_id`
- Conditioning: `e_diff <- e_diff + e_f` (default) or finding as 4th token
- Output: `d_f` (512-d change emb for **this** finding on **this** pair) + optional `mag`

In [ ]:
class DifferenceTransformer(nn.Module):
    """Finding-conditioned temporal difference module. d_f = g(vp, vc, finding)."""

    def __init__(self, n_findings=18, d_in=512, d_model=256, n_layers=2, n_heads=4,
                 dropout=0.1, antisym=False, magnitude=False,
                 finding_conditioning=True, finding_as_4th_token=False,
                 use_learned_finding_emb=True, frozen_finding_emb=None,
                 tau_con_init=0.07):
        super().__init__()
        self.finding_conditioning = finding_conditioning
        self.finding_as_4th_token = finding_as_4th_token and finding_conditioning
        self.antisym = antisym
        self.W = nn.Linear(d_in, d_model)
        self.role = nn.Parameter(torch.randn(2, d_model) * 0.02)
        self.e_diff = nn.Parameter(torch.randn(1, d_model) * 0.02)
        if finding_conditioning:
            if use_learned_finding_emb:
                self.finding_emb = nn.Embedding(n_findings, d_model)
                nn.init.normal_(self.finding_emb.weight, std=0.02)
            else:
                assert frozen_finding_emb is not None
                self.register_buffer('finding_name_512', frozen_finding_emb.float())
                self.finding_proj = nn.Linear(d_in, d_model)
                self.finding_emb = None
        else:
            self.finding_emb = None
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * 4, dropout=dropout,
            batch_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d_model, d_in)
        self.mag_head = nn.Linear(d_model, 1) if magnitude else None
        self.logit_scale = nn.Parameter(torch.tensor(float(np.log(1 / 0.07))))
        self.log_tau_con = nn.Parameter(torch.tensor(float(np.log(tau_con_init))))

    def _e_f(self, fid):
        if not self.finding_conditioning:
            return None
        if self.finding_emb is not None:
            return self.finding_emb(fid)
        return self.finding_proj(self.finding_name_512[fid])

    def _pass(self, vp, vc, fid):
        B = vp.size(0)
        tp = self.W(vp) + self.role[0]
        tc = self.W(vc) + self.role[1]
        ed = self.e_diff.expand(B, -1).clone()
        ef = self._e_f(fid)
        if ef is not None and not self.finding_as_4th_token:
            ed = ed + ef
            seq = torch.stack([ed, tp, tc], dim=1)
        elif ef is not None and self.finding_as_4th_token:
            seq = torch.stack([ed, tp, tc, ef], dim=1)
        else:
            seq = torch.stack([ed, tp, tc], dim=1)
        h = self.enc(seq)
        hdiff = h[:, 0]
        mag = self.mag_head(hdiff).squeeze(-1) if self.mag_head is not None else None
        return self.head(hdiff), mag

    def forward(self, vp, vc, fid=None):
        if fid is None:
            fid = torch.zeros(vp.size(0), dtype=torch.long, device=vp.device)
        vd, mag = self._pass(vp, vc, fid)
        if self.antisym:
            vd_rev, _ = self._pass(vc, vp, fid)
            vd = vd - vd_rev
        return vd, mag

    def tau_con(self):
        return self.log_tau_con.exp().clamp(min=1e-3, max=1.0)


def logits_from(vd, proto, logit_scale):
    vd = F.normalize(vd, dim=-1)
    pr = F.normalize(proto, dim=-1)
    cos = torch.einsum('bd,bkd->bk', vd, pr)
    return logit_scale.exp().clamp(max=100) * cos


def masked_supcon_loss(z, y, fid, tau):
    """Same-finding masked SupCon: + same dir, - other dir, ignore other findings."""
    z = F.normalize(z, dim=-1)
    B = z.size(0)
    if B < 2:
        return z.new_zeros(())
    sim = (z @ z.t()) / tau
    self_mask = torch.eye(B, dtype=torch.bool, device=z.device)
    same_f = fid.unsqueeze(0).eq(fid.unsqueeze(1)) & ~self_mask
    pos_mask = same_f & y.unsqueeze(0).eq(y.unsqueeze(1))
    allowed = same_f
    pos_counts = pos_mask.sum(dim=1).float()
    valid = (pos_counts > 0) & (allowed.sum(dim=1) > 0)
    if not valid.any():
        return z.new_zeros(())
    neg_large = torch.finfo(sim.dtype).min / 2
    logits = sim.masked_fill(self_mask | ~allowed, neg_large)
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    exp_logits = logits.exp() * allowed.float()
    log_prob = logits - exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-8).log()
    pos_log = torch.where(pos_mask, log_prob, torch.zeros_like(log_prob))
    mean_pos = pos_log.sum(dim=1) / pos_counts.clamp(min=1.0)
    loss = -mean_pos[valid].mean()
    return loss if torch.isfinite(loss) else z.new_zeros(())


npar = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
_probe = DifferenceTransformer(
    n_findings=len(FINDINGS), d_model=D_MODEL, antisym=ANTISYM, magnitude=USE_MAGNITUDE,
    finding_conditioning=FINDING_CONDITIONING, finding_as_4th_token=FINDING_AS_4TH_TOKEN,
    use_learned_finding_emb=USE_LEARNED_FINDING_EMB,
    frozen_finding_emb=None if USE_LEARNED_FINDING_EMB else FINDING_NAME_EMB,
    tau_con_init=TAU_CON_INIT,
)
print('module trainable params:', f'{npar(_probe):,}')
del _probe

## 9. Contrastive-aware batch sampler

Build batches that cover ~`K_FINDINGS_PER_BATCH` findings with rows from multiple direction classes so masked SupCon has same-finding positives **and** other-direction negatives.

In [ ]:
def build_buckets(split):
    """Map (fid, y) -> list of dataset indices for contrastive-aware sampling."""
    y = DATA[split]['y']; fid = DATA[split]['fid']
    buckets = defaultdict(list)
    for i in range(len(y)):
        buckets[(int(fid[i]), int(y[i]))].append(i)
    return buckets

TRAIN_BUCKETS = build_buckets('train')
print('train buckets (fid,y) with data:', len(TRAIN_BUCKETS),
      '| findings present:', len({k[0] for k in TRAIN_BUCKETS}))

def sample_contrastive_batch(buckets, k_findings=K_FINDINGS_PER_BATCH,
                             n_per_class=N_PER_CLASS, max_bs=MAX_BATCH_SIZE, rng=None):
    rng = rng or random
    by_f = defaultdict(set)
    for (f, y), idxs in buckets.items():
        if idxs:
            by_f[f].add(y)
    eligible = [f for f, ys in by_f.items() if len(ys) >= 2]
    if not eligible:
        eligible = list(by_f.keys())
    if not eligible:
        return []
    k = min(k_findings, len(eligible))
    chosen_f = rng.sample(eligible, k)
    batch = []
    for f in chosen_f:
        for y in range(3):
            pool = buckets.get((f, y), [])
            if not pool:
                continue
            take = min(n_per_class, len(pool))
            batch.extend(rng.sample(pool, take) if len(pool) >= take else list(pool))
    if not batch:
        return []
    if len(batch) > max_bs:
        batch = rng.sample(batch, max_bs)
    rng.shuffle(batch)
    return batch

def steps_per_epoch(n, approx_bs):
    return max(1, math.ceil(n / max(approx_bs, 1)))

_bs_probe = [len(sample_contrastive_batch(TRAIN_BUCKETS)) for _ in range(20)]
print('probe batch sizes: min/mean/max =', min(_bs_probe),
      round(sum(_bs_probe)/len(_bs_probe), 1), max(_bs_probe))
APPROX_BS = max(int(sum(_bs_probe)/len(_bs_probe)), 32)
STEPS = steps_per_epoch(len(DATA['train']['y']), APPROX_BS)
print('steps/epoch ~', STEPS, '| approx_bs ~', APPROX_BS)

## 10. Train on Hub `train_*` (early-stop on patient-held-out `train_*` tune)

Total loss:

**L = λ_ce L_CE + λ_mag L_mag + λ_con L_con**

- `L_CE` — weighted CE on `cosine(d_f, PROTO[f]) * exp(logit_scale)`
- `L_mag` — optional BCE(mag, change vs stable)
- `L_con` — masked same-finding SupCon with temperature `tau_con`

In [ ]:
model = DifferenceTransformer(
    n_findings=len(FINDINGS),
    d_model=D_MODEL,
    antisym=ANTISYM,
    magnitude=USE_MAGNITUDE,
    finding_conditioning=FINDING_CONDITIONING,
    finding_as_4th_token=FINDING_AS_4TH_TOKEN,
    use_learned_finding_emb=USE_LEARNED_FINDING_EMB,
    frozen_finding_emb=None if USE_LEARNED_FINDING_EMB else FINDING_NAME_EMB,
    tau_con_init=TAU_CON_INIT,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce_loss_fn = nn.CrossEntropyLoss(weight=W_cls.to(DEVICE))

@torch.no_grad()
def evaluate(split):
    model.eval()
    D = DATA[split]
    all_y, all_p = [], []
    tot, n = 0.0, 0
    bs = 512
    for i in range(0, len(D['y']), bs):
        sl = slice(i, i + bs)
        vp = D['vp'][sl].to(DEVICE); vc = D['vc'][sl].to(DEVICE)
        pr = D['pr'][sl].to(DEVICE); y  = D['y'][sl].to(DEVICE)
        fid = D['fid'][sl].to(DEVICE)
        vd, mag = model(vp, vc, fid)
        lg = logits_from(vd, pr, model.logit_scale)
        loss = ce_loss_fn(lg, y)
        tot += loss.item() * len(y); n += len(y)
        all_y += y.cpu().tolist()
        all_p += lg.argmax(1).cpu().tolist()
    mf1 = f1_score(all_y, all_p, labels=[0, 1, 2], average='macro', zero_division=0)
    return tot / max(n, 1), mf1, np.array(all_y), np.array(all_p)

def train_epoch():
    model.train()
    tot, n = 0.0, 0
    all_y, all_p = [], []
    for _ in range(STEPS):
        if USE_SUPCON:
            idxs = sample_contrastive_batch(TRAIN_BUCKETS)
            if len(idxs) < 4:
                idxs = random.sample(
                    range(len(DATA['train']['y'])),
                    min(MAX_BATCH_SIZE, len(DATA['train']['y'])))
        else:
            idxs = random.sample(
                range(len(DATA['train']['y'])),
                min(MAX_BATCH_SIZE, len(DATA['train']['y'])))
        idxs_t = torch.tensor(idxs, dtype=torch.long)
        vp = DATA['train']['vp'][idxs_t].to(DEVICE)
        vc = DATA['train']['vc'][idxs_t].to(DEVICE)
        pr = DATA['train']['pr'][idxs_t].to(DEVICE)
        y  = DATA['train']['y'][idxs_t].to(DEVICE)
        fid = DATA['train']['fid'][idxs_t].to(DEVICE)

        vd, mag = model(vp, vc, fid)
        lg = logits_from(vd, pr, model.logit_scale)

        loss = vd.new_zeros(())
        if USE_CE:
            loss = loss + LAMBDA_CE * ce_loss_fn(lg, y)
        if USE_MAGNITUDE and mag is not None:
            is_change = (y != C2I['stable']).float()
            loss = loss + LAMBDA_MAG * F.binary_cross_entropy_with_logits(mag, is_change)
        if USE_SUPCON:
            loss = loss + LAMBDA_CON * masked_supcon_loss(vd, y, fid, model.tau_con())

        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        tot += loss.item() * len(idxs); n += len(idxs)
        all_y += y.cpu().tolist()
        all_p += lg.argmax(1).cpu().tolist()
    mf1 = f1_score(all_y, all_p, labels=[0, 1, 2], average='macro', zero_division=0)
    return tot / max(n, 1), mf1

best, best_state, bad = -1.0, None, 0
for ep in range(1, EPOCHS + 1):
    tl, _ = train_epoch()
    _, vf1, _, _ = evaluate('tune')
    if vf1 > best:
        best = vf1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if ep % 10 == 0 or ep == 1:
        print(f'ep {ep:3}  loss {tl:.3f}  tune_macroF1 {vf1:.3f}  (best {best:.3f})  '
              f'tau_con={model.tau_con().item():.4f}  '
              f'logit_scale={model.logit_scale.exp().item():.2f}')
    if bad >= PATIENCE:
        print(f'early stop @ ep {ep}  best tune macro-F1 {best:.3f}')
        break

assert best_state is not None
model.load_state_dict(best_state)
print('restored best. train-domain tune macro-F1 =', round(best, 3))
print('flags:', dict(
    FINDING_CONDITIONING=FINDING_CONDITIONING, FINDING_AS_4TH_TOKEN=FINDING_AS_4TH_TOKEN,
    USE_MAGNITUDE=USE_MAGNITUDE, USE_SUPCON=USE_SUPCON, PROTO_SOURCE=PROTO_SOURCE,
    ANTISYM=ANTISYM, LAMBDA_MAG=LAMBDA_MAG, LAMBDA_CON=LAMBDA_CON))

## 11. FINAL TEST — CT-RATE Hub `valid_*` only

Run **once** after architecture / hyperparameters are fixed. Do not tune from these numbers.

Inference: `d_f = g(v_p, v_c, f)` -> `argmax cos(d_f, PROTO[f])` — **no report text**.

In [ ]:
_, macro, y, pred = evaluate('test')
percls = f1_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
acc = (y == pred).mean()
maj = Counter(DATA['train']['y'].tolist()).most_common(1)[0][0]
maj_macro = f1_score(y, np.full_like(y, maj), labels=[0, 1, 2], average='macro', zero_division=0)
assert all(e['hub_domain'] == 'hub_valid' for e in examples['test'])
print('=== FINAL HUB-VALIDATION TEST — per class ===')
print(f'accuracy : {acc:.3f}')
print(f'macro-F1 : {macro:.3f}   (always-{CLASSES[maj]} ref = {maj_macro:.3f})')
for i, c in enumerate(CLASSES):
    print(f'  F1 {c:9}: {percls[i]:.3f}')
print('\nconfusion (rows=true, cols=pred; order w/s/i):')
print(confusion_matrix(y, pred, labels=[0, 1, 2]))

## 12. FINAL HUB-VALIDATION TEST — per disease (finding)

In [ ]:
model.eval()
D = DATA['test']
all_p = []
with torch.no_grad():
    bs = 512
    for i in range(0, len(D['y']), bs):
        sl = slice(i, i + bs)
        vd, _ = model(D['vp'][sl].to(DEVICE), D['vc'][sl].to(DEVICE), D['fid'][sl].to(DEVICE))
        lg = logits_from(vd, D['pr'][sl].to(DEVICE), model.logit_scale)
        all_p.append(lg.argmax(1).cpu())
pred = torch.cat(all_p).numpy()
y = D['y'].numpy()
Fn = D['fn']
by_f = defaultdict(lambda: {'y': [], 'p': []})
for yi, pi, fi in zip(y, pred, Fn):
    by_f[fi]['y'].append(yi); by_f[fi]['p'].append(pi)
rows = []
for f, d in by_f.items():
    yy, pp = np.array(d['y']), np.array(d['p'])
    present = sorted(set(yy.tolist()))
    rows.append((f, len(yy), (yy == pp).mean(),
                 f1_score(yy, pp, labels=present, average='macro', zero_division=0)))
rows.sort(key=lambda r: -r[1])
print('=== FINAL HUB-VALIDATION TEST — per disease ===')
print(f"{'finding':<34}{'n':>5}{'acc':>7}{'macroF1*':>10}")
for f, n, a, mf1 in rows:
    print(f'{f:<34}{n:>5}{a:>7.3f}{mf1:>10.3f}')
print('\n* macro-F1 over classes actually present for that finding')

## 13. Save checkpoint

In [ ]:
ckpt_dir = f'{DRIVE}/ctclip_cache/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
tag = (
    f"fc{int(FINDING_CONDITIONING)}_mag{int(USE_MAGNITUDE)}_supcon{int(USE_SUPCON)}"
    f"_proto{PROTO_SOURCE}_seed{SPLIT_SEED}"
)
ckpt_path = f'{ckpt_dir}/proposed_{tag}.pt'
torch.save({
    'model': best_state,
    'tune_macro_f1': best,
    'findings': FINDINGS,
    'classes': CLASSES,
    'config': dict(
        FINDING_CONDITIONING=FINDING_CONDITIONING,
        FINDING_AS_4TH_TOKEN=FINDING_AS_4TH_TOKEN,
        USE_LEARNED_FINDING_EMB=USE_LEARNED_FINDING_EMB,
        USE_MAGNITUDE=USE_MAGNITUDE,
        USE_SUPCON=USE_SUPCON,
        PROTO_SOURCE=PROTO_SOURCE,
        ANTISYM=ANTISYM,
        D_MODEL=D_MODEL, LR=LR,
        LAMBDA_CE=LAMBDA_CE, LAMBDA_MAG=LAMBDA_MAG, LAMBDA_CON=LAMBDA_CON,
        TAU_CON_INIT=TAU_CON_INIT,
        K_FINDINGS_PER_BATCH=K_FINDINGS_PER_BATCH,
        N_PER_CLASS=N_PER_CLASS,
    ),
}, ckpt_path)
print('saved', ckpt_path)

## Notes / ablations

| Knob | Suggested sweeps |
|---|---|
| `FINDING_CONDITIONING` | `True` (proposed) vs `False` (current shared-`d` baseline) |
| `FINDING_AS_4TH_TOKEN` | `False` (`e_diff+e_f`) vs `True` (4th token) |
| `USE_MAGNITUDE` | off / on (`LAMBDA_MAG in {0.25, 0.5, 1.0}`) |
| `USE_SUPCON` | off / on (`LAMBDA_CON in {0.1, 0.25, 0.5}`) |
| `PROTO_SOURCE` | `templates` vs `real` |
| `USE_LEARNED_FINDING_EMB` | learned `nn.Embedding` vs frozen text-name emb |

**Loss design reminders**
- SupCon is **same-finding masked** — other findings never appear as negatives (avoids punishing cross-finding geometry).
- `tau_con` is **separate** from CE `logit_scale` (avoids temperature tug-of-war).
- Magnitude head is an **aux regularizer** only; inference is still pure prototype argmax.

**Protocol (do not break)**
- Optimize + early-stop on Hub `train_*` only (`tune` = patient-held-out fraction).
- Inspect Hub `valid_*` **once** after freezing choices.
- Inference never uses report text — only `v_prior`, `v_current`, `finding` id/name.

**Compare to current notebook** (`train_ctclip_temporal_colab.ipynb`):
- Current: `d = g(v_p, v_c)` shared across findings; optional InfoNCE(`d`, report text).
- Proposed: `d_f = g(v_p, v_c, f)`; masked SupCon on directions within a finding.

See `docs/proposed_pipeline.png` and `docs/current_pipeline.png`.